# Projeto (Seção 2) — solução comentada

Solução própria para este repositório: modelo, testes do enunciado, **testes extras** (casos de erro esperados), revisão da sua
tentativa (`11 - My_Project` do material original) e comparação com a solução do instrutor.

Enunciado em [`11_projeto`](11_projeto.ipynb).

## 1. Solução

In [1]:
from datetime import date
from pydantic import BaseModel, ValidationError


class Automobile(BaseModel):
    manufacturer: str
    series_name: str
    type_: str
    is_electric: bool = False
    manufactured_date: date
    base_msrp_usd: float
    vin: str
    number_of_doors: int = 4
    registration_country: str | None = None    # opcional E anulável
    license_plate: str | None = None           # opcional E anulável

## 2. Testes do enunciado

In [2]:
data = {
    "manufacturer": "BMW", "series_name": "M4", "type_": "Convertible", "is_electric": False,
    "manufactured_date": "2023-01-01", "base_msrp_usd": 93_300, "vin": "1234567890",
    "number_of_doors": 2, "registration_country": "France", "license_plate": "AAA-BBB",
}
data_expected_serialization = {
    'manufacturer': 'BMW', 'series_name': 'M4', 'type_': 'Convertible', 'is_electric': False,
    'manufactured_date': date(2023, 1, 1), 'base_msrp_usd': 93_300, 'vin': '1234567890',
    'number_of_doors': 2, 'registration_country': 'France', 'license_plate': 'AAA-BBB',
}
data_json = '''{"manufacturer": "BMW", "series_name": "M4", "type_": "Convertible",
                "manufactured_date": "2023-01-01", "base_msrp_usd": 93300, "vin": "1234567890"}'''
data_json_expected_serialization = {
    'manufacturer': 'BMW', 'series_name': 'M4', 'type_': 'Convertible', 'is_electric': False,
    'manufactured_date': date(2023, 1, 1), 'base_msrp_usd': 93_300, 'vin': '1234567890',
    'number_of_doors': 4, 'registration_country': None, 'license_plate': None,
}

car = Automobile.model_validate(data)
assert car.model_dump() == data_expected_serialization
car_json = Automobile.model_validate_json(data_json)
assert car_json.model_dump() == data_json_expected_serialization
print(f"car: {car!r}")
print(f"car_json.model_fields_set: {sorted(car_json.model_fields_set)}")
print("testes do enunciado: ok")

car: Automobile(manufacturer='BMW', series_name='M4', type_='Convertible', is_electric=False, manufactured_date=datetime.date(2023, 1, 1), base_msrp_usd=93300.0, vin='1234567890', number_of_doors=2, registration_country='France', license_plate='AAA-BBB')
car_json.model_fields_set: ['base_msrp_usd', 'manufactured_date', 'manufacturer', 'series_name', 'type_', 'vin']
testes do enunciado: ok


## 3. Testes extras

O enunciado só testa o caminho feliz. Os casos abaixo confirmam as regras de obrigatoriedade, anulabilidade e coerção.

In [3]:
def expect_errors(data: dict) -> list[tuple]:
    """Valida esperando falha; devolve (campo, tipo_do_erro) de cada erro."""
    try:
        Automobile.model_validate(data)
    except ValidationError as ex:
        return [(err["loc"][0], err["type"]) for err in ex.errors()]
    raise AssertionError("era esperado ValidationError")

# obrigatórios ausentes: todos reportados de uma vez
missing = expect_errors({})
print(f"campos obrigatórios ausentes: {missing}")
assert {f for f, t in missing if t == "missing"} == {
    "manufacturer", "series_name", "type_", "manufactured_date", "base_msrp_usd", "vin"}

# não anulável recebendo None
print(f"is_electric=None: {expect_errors({**data, 'is_electric': None})}")

# anuláveis aceitam None explicitamente
explicit_none = Automobile.model_validate({**data, "registration_country": None, "license_plate": None})
assert explicit_none.registration_country is None and explicit_none.license_plate is None

# coerção lax: "2" -> 2, "93300.5" -> 93300.5; e o que não converte
coerced = Automobile.model_validate({**data, "number_of_doors": "2", "base_msrp_usd": "93300.5"})
print(f"coerced.number_of_doors, coerced.base_msrp_usd: {coerced.number_of_doors!r}, {coerced.base_msrp_usd!r}")
print(f"number_of_doors=2.5: {expect_errors({**data, 'number_of_doors': 2.5})}")
print(f"manufactured_date='01/01/2023': {expect_errors({**data, 'manufactured_date': '01/01/2023'})}")
print("testes extras: ok")

campos obrigatórios ausentes: [('manufacturer', 'missing'), ('series_name', 'missing'), ('type_', 'missing'), ('manufactured_date', 'missing'), ('base_msrp_usd', 'missing'), ('vin', 'missing')]
is_electric=None: [('is_electric', 'bool_type')]
coerced.number_of_doors, coerced.base_msrp_usd: 2, 93300.5
number_of_doors=2.5: [('number_of_doors', 'int_from_float')]
manufactured_date='01/01/2023': [('manufactured_date', 'date_from_datetime_parsing')]
testes extras: ok


## 4. Revisão da sua tentativa

Sua solução (`11 - My_Project`) **passa** nos dois testes do enunciado. Um ponto a corrigir:

```python
registration_country: str = None      # sua versão
license_plate: str = None
```

Discordo dessa forma porque o *type hint* diz "string não anulável", mas o default é `None`: funciona só porque o Pydantic não valida
defaults. O que eu faria no lugar: `str | None = None`, que declara o campo **opcional e anulável** (é o que o enunciado pede —
default `None`). O risco do seu caminho é concreto: o payload `{"registration_country": null}` (enviar `null` explicitamente) é
**rejeitado**, embora o modelo crie instâncias com `None` sozinho. A célula abaixo prova:

In [4]:
class AutomobileAttempt(BaseModel):      # sua versão, campos relevantes
    vin: str
    registration_country: str = None

print(f'AutomobileAttempt(vin="1"): {AutomobileAttempt(vin="1")!r}')
try:
    AutomobileAttempt(vin="1", registration_country=None)
except ValidationError as ex:
    print(f"registration_country=None explícito: {ex.errors()[0]['type']}")

AutomobileAttempt(vin="1"): AutomobileAttempt(vin='1', registration_country=None)
registration_country=None explícito: string_type


## 5. Comparação com a solução do instrutor

| ponto | instrutor | esta solução |
|---|---|---|
| campos e tipos | idênticos | idênticos |
| anuláveis | `str \| None = None` | `str \| None = None` |
| testes | só caminho feliz | + obrigatórios, `None` explícito, coerção e erros de formato |

A modelagem é a mesma — a diferença está na cobertura de testes. `base_msrp_usd: float` com dinheiro é aceitável neste estágio do
curso; em produção, `Decimal` evita erros de arredondamento.